# PGMIDD Demo

**Pipeline:** Upload image → Qwen VL (auto-describe) → GroundingDINO (detect) → Annotated output + Q&A

**Models:**
- GroundingDINO Base (SwinT OGC) — open-vocabulary detection
- Qwen2.5-VL-7B fine-tuned — description generation + Q&A



In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/AIP491

Mounted at /content/drive
/content/drive/MyDrive/AIP491


In [2]:
# Clone GroundingDINO + install dependencies
!git clone https://github.com/IDEA-Research/GroundingDINO.git /workspace/GroundingDINO
%cd /workspace/GroundingDINO
!pip install -e . 2>&1 | tail -3
%cd /content/drive/MyDrive/AIP491

Cloning into '/workspace/GroundingDINO'...
remote: Enumerating objects: 463, done.
remote: Total 463 (delta 0), reused 0 (delta 0), pack-reused 463 (from 1)
Receiving objects: 100% (463/463), 12.91 MiB | 11.80 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/workspace/GroundingDINO
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
/content/drive/MyDrive/AIP491


In [3]:
!mkdir -p /workspace/weights
!wget -q -O /workspace/weights/groundingdino_swint_ogc.pth \
"https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth"
!ls -lh /workspace/weights/groundingdino_swint_ogc.pth

-rw-r--r-- 1 root root 662M Mar 21  2023 /workspace/weights/groundingdino_swint_ogc.pth


In [4]:
!pip install -q torch torchvision gradio
!pip install -q bitsandbytes
!pip install -q "transformers>=4.41.0,<5.0.0" accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00


In [5]:
import os, sys
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WEIGHTS_DIR"]   = "/workspace/weights"
os.environ["GDINO_ROOT"]   = "/workspace/GroundingDINO"

print(f"WEIGHTS_DIR : {os.environ['WEIGHTS_DIR']}")
print(f"GDINO_ROOT  : {os.environ['GDINO_ROOT']}")

WEIGHTS_DIR : /workspace/weights
GDINO_ROOT  : /workspace/GroundingDINO


In [6]:
import transformers
import torch

_orig = transformers.cache_utils.StaticCache.update

def _patched_update(self, key_states, value_states, max_cache_length,
                     cache_position, kv_permissions_mask, **_kwargs):
    key_states   = key_states.to(torch.bfloat16)
    value_states = value_states.to(torch.bfloat16)
    return _orig(self, key_states, value_states, max_cache_length,
                 cache_position, kv_permissions_mask, **_kwargs)

transformers.cache_utils.StaticCache.update = _patched_update
print(f"StaticCache patched. GPU: {torch.cuda.get_device_name(0)}")

StaticCache patched. GPU: NVIDIA L4


## Gradio Demo

# Cell 8 — SKIP (patches moved into gradio_demo.py)
# gradio_demo.py now applies all patches automatically on startup.
# No monkey-patching needed here. Jump straight to cell-9.

In [ ]:
import torch

!python /content/drive/MyDrive/AIP491/demo/gradio_demo.py --port 7860 --share

[Demo] PGMIDD — port 7860 | device cuda
[Demo] Qwen VL: enabled
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://a8cc78db66895d14ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
2026-04-09 07:51:53.231593: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 07:51:53.250662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775721113.274708    6019 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register f